In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

from sklearn.metrics import mean_absolute_error
# Download a real housing dataset from Scikit-Learn
housing = fetch_california_housing(as_frame=True)

# Put it into a Pandas DataFrame (which is basically an Excel sheet for Python)
df = housing.frame

# Show the first 5 rows of the data  // df.head shows first 5 columns
df.head()

In [ ]:
import matplotlib.pyplot as plt

# The 'bins' dictate how granular the bar chart is. 
# figsize makes it large enough to read. 
# we gonna see that some data is "capped" so no max value is shown. ex. HOUSE VALUE is capped at 500k

df.hist(bins=50, figsize=(20, 15))
plt.show()

In [ ]:
#  exactly how many neighborhoods hit that cap
capped_count = (df['MedHouseVal'] >= 5.0).sum()
print(f"Number of capped neighborhoods: {capped_count}  notice now we have more clever data points to work with")

# Keeps only the rows where the price is STRICTLY LESS than 5.0
df = df[df['MedHouseVal'] < 5.0]

#  histogram  to prove it is fixed
df['MedHouseVal'].hist(bins=50, figsize=(8, 5))
plt.title("Median House Value (Caps Removed)")
plt.show()

In [ ]:
import numpy as np

# Improvent in  data trying to make it more "AI-friendly" 
#         by creating new features and applying log transforms to the tail-heavy columns.

#Create the new ratio feature 
#more rooms then bedrooms =  expensive 
df['Bedrms_per_Room'] = df['AveBedrms'] / df['AveRooms']

# Apply Log Transform to  tail-heavy columns
# use np.log() to squash the massive outliers into a normal bell curve
df['Population_Log'] = np.log(df['Population'])
df['AveOccup_Log'] = np.log(df['AveOccup'])
df['AveRooms_Log'] = np.log(df['AveRooms'])

# Drop the old, un-squashed columns so they don't confuse the AI
df = df.drop(['Population', 'AveOccup', 'AveRooms', 'AveBedrms'], axis=1)

print("Data improvements applied! Look at the new columns:")
display(df.head())
# after this change we gonna have a slightly better model performance...

In [ ]:
#Check how many rows/columns we have and if any data is missing
df.info()

print("\n-------------------\n")

#Get the mathematical summary (average age, max price, etc.)
df.describe()

In [ ]:
#income of the area, the age of the house, and how many rooms it has without MedHouseVal.
X = df.drop('MedHouseVal',axis =1)

# y is the answer we want to predict
y = df['MedHouseVal']

print("Clues (X) shape:", X.shape)
print("Answers (y) shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data: 80% for training, 20% for testing, 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total neighborhoods: {len(X)}")
print(f"Training set (Textbook): {len(X_train)}")
print(f"Testing set (Final Exam): {len(X_test)}")

In [ ]:
# Now we have our training and testing sets, we can start to build a model. 
# But first, let's scale the features so they're all on the same level playing field.
from sklearn.preprocessing import StandardScaler

# Create a scaler object
scaler = StandardScaler()

# Fit on training data ONLY, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features have been successfully scaled!")

In [ ]:
from sklearn.linear_model import LinearRegression

#Create the empty 'brain'
model = LinearRegression()

#Training: Give the brain the Textbook (X_train) and the Answers (y_train)
model.fit(X_train_scaled, y_train)

print("The AI has finished studying!")

# Ask the AI to guess the prices using X_test_scaled!
predictions = model.predict(X_test_scaled)

#Compare the first 5 guesses to the actual real prices
print("AI's first 5 Guesses: ", predictions[:5])
print("Actual Real Prices:  ", y_test[:5].values)

# Score the model using X_test_scaled!
score = model.score(X_test_scaled, y_test)
print(f"The AI's Accuracy Score (R-squared) is: {score:.2%}")